<a href="https://colab.research.google.com/github/shivamagarwalhere/practice-repository/blob/main/Study_Assistant_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Study Assistant for Quiz Question Generation

We need `langchain-google-genai` to interface with Gemini and `PyPDF2` for document processing.

In [1]:
!pip install -q -U langchain-google-genai PyPDF2 langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 6.1 MB/s eta 0:00:00


In [32]:
import os
import getpass
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Securely fetch the API key from Colab secrets or manual input
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    print("'GOOGLE_API_KEY' not found in Secrets. Please enter it below:")
    GOOGLE_API_KEY = getpass.getpass("Enter your Gemini API Key: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Initialize the Gemini Model
try:
    # Using the exact identifier from genai.list_models(): models/gemini-flash-latest
    llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0.3)
    print("Gemini Model Initialized.")
except Exception as e:
    print(f"Initialization Error: {e}")

'GOOGLE_API_KEY' not found in Secrets. Please enter it below:
Enter your Gemini API Key: ··········
Gemini Model Initialized.


In [18]:
import google.generativeai as genai
genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [33]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    """Extracts all text from a given PDF file."""
    try:
        with open(pdf_path, "rb") as file:
            reader = PyPDF2.PdfReader(file)
            text = ""
            for page in reader.pages:
                content = page.extract_text()
                if content:
                    text += content + "\n"
            return text.strip()
    except FileNotFoundError:
        return "Error: PDF file not found. Please upload it to Colab."

study_material = extract_text_from_pdf("/content/Prompt_Engineering.pdf")
print(study_material[:500]) # Print first 500 characters

What
is
Prompt
Engineering?
Prompt
engineering
is
a
practice
within
natural
language
processing
(NLP)
in
artificial
intelligence,
where
text
is
used
to
describe
the
task
the
AI
should
perform.
Guided
by
this
input,
the
AI
generates
an
output,
which
could
take
various
forms.
The
goal
is
to
use
human-understandable
text
to
interact
conversationally
with
models,
allowing
for
flexibility
in
the
model’ s
performance
due
to
the
task
description
embedded
in
the
prompt.
What
are
Prompts?
Prompts
are
det


We will use LangChain's `PromptTemplate` to structure our requests to Gemini.

In [34]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Summarization Prompt
summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Summarize the following study material into concise, high-level bullet points:

{text}

Summary:"""
)

# 2. Quiz Generation Prompt
quiz_prompt = PromptTemplate(
    input_variables=["summary"],
    template="""Based on the following summary, generate 3-5 multiple-choice questions.
Each question must have 4 options (a, b, c, d) and provide the correct answer at the end of each question.

Summary:
{summary}

Quiz Questions:"""
)

# Create chains with string output parsers
summary_chain = summary_prompt | llm | StrOutputParser()
quiz_chain = quiz_prompt | llm | StrOutputParser()

print("Chains Ready.")

Chains Ready.


In [36]:
from IPython.display import display, Markdown

def run_study_assistant(pdf_path):
    # 1. Extract
    print(f"Reading PDF: {pdf_path}...")
    raw_text = extract_text_from_pdf(pdf_path)
    if "❌" in raw_text:
        print(raw_text)
        return

    # 2. Summarize
    print("\n--- GENERATING SUMMARY ---")
    summary_text = summary_chain.invoke({"text": raw_text[:10000]})
    print(summary_text)

    # 3. Generate Quiz
    print("\n--- GENERATING QUIZ ---")
    quiz_text = quiz_chain.invoke({"summary": summary_text})
    print(quiz_text)

Now we execute the pipeline using the uploaded PDF. This will extract the text, generate a summary, and create a quiz.

In [37]:
run_study_assistant("/content/Prompt_Engineering.pdf")

Reading PDF: /content/Prompt_Engineering.pdf...

--- GENERATING SUMMARY ---
Here is a concise, high-level summary of the study material:

*   **Definition**: Prompt engineering is an NLP practice where human-understandable text is used to describe tasks and guide AI models to generate specific outputs.
*   **Core Function**: Prompts serve as the interface between the user and the model, providing the detailed instructions necessary to define the AI's expected behavior.
*   **Versatility**: Prompting is used across various mediums, including **text** (summaries, creative writing), **code** (debugging, generation), and **images** (artistic styles, specific visualizations).
*   **Key Elements of a Prompt**:
    *   **Instruction**: The specific task to perform.
    *   **Context**: Background information to frame the task.
    *   **Input Data**: The actual data the model needs to process.
    *   **Output Indicator**: The desired format or type of response.
*   **Optimization Strategies*